In [127]:
import pandas as pd
import glob
import sys
import subprocess

## Step 1: Automated Data Discovery and Merging
In this step, we dynamically locate all monthly sales CSV files using the `glob` library and merge them into a single comprehensive DataFrame. This approach ensures scalability, allowing the pipeline to automatically handle any number of incoming monthly reports without changing the source code.

In [128]:
# 1. Dynamically find all monthly sales files
file_paths = glob.glob("large_sales_month*.csv")
print(f"Files found for merging: {file_paths}")

Files found for merging: ['large_sales_month1.csv', 'large_sales_month2.csv', 'large_sales_month3.csv']


In [129]:
# 2. Read each CSV file into a list of DataFrames
df_list = [pd.read_csv(file) for file in file_paths]

In [130]:
# 3. Concatenate all DataFrames into a single unified dataset
df = pd.concat(df_list, ignore_index=True)
print(f"Total rows after merging: {df.shape[0]}\n")

Total rows after merging: 15000



In [131]:
# 4. Preliminary dataset inspection
print("--- Dataset General Information ---")
print(df.info())

print("\n--- Missing Values Count Per Column ---")
print(df.isna().sum())

print("\n--- First 5 Rows of the Merged Dataset ---")
print(df.head(5))

--- Dataset General Information ---
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sale_id       15000 non-null  int64  
 1   date          15000 non-null  str    
 2   product_name  14518 non-null  str    
 3   price         14579 non-null  str    
 4   quantity      14723 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 586.1 KB
None

--- Missing Values Count Per Column ---
sale_id           0
date              0
product_name    482
price           421
quantity        277
dtype: int64

--- First 5 Rows of the Merged Dataset ---
   sale_id              date    product_name    price  quantity
0    10000  2026/01/04 18:00     MacBook Pro     1999       1.0
1    10001  2026/01/20 15:00    macbook air      1100       3.0
2    10002        07/01/2026       iPad Air   599 USD       4.0
3    10003  2026/01/11 07:00        iPad Air   

## Step 2: Data Cleaning and Type Casting
In this step, we standardize the text columns, clean up the price formats by removing currency symbols (`$`, `USD`), handle missing values ($NaN$), and cast all columns to their appropriate analytical data types (`float`, `int`, etc.). Missing prices are intelligently imputed based on the median price of each respective product.

In [132]:
# 1. Clean and standardize Product Names
# Trim whitespaces and capitalize correctly (e.g., lowercase 'iphone' becomes standard)
df['product_name'] = df['product_name'].str.strip()
# Let's fix the case inconsistencies by grouping them properly later, 
# but first fill missing product names with 'Unknown Product'
df['product_name'] = df['product_name'].fillna('Unknown Product')

In [133]:
# 2. Clean Price Column and Convert to Numeric
# Remove characters like $, USD, commas, and spaces using Regex
df['price_cleaned'] = df['price'].astype(str).str.replace(r'[\$,\s|USD]', '', regex=True)
# Convert to float, turning errors or empty values into NaN
df['price_cleaned'] = pd.to_numeric(df['price_cleaned'], errors='coerce')

# Intelligently fill missing prices using the median price of that specific product
df['price_cleaned'] = df.groupby('product_name')['price_cleaned'].transform(lambda x: x.fillna(x.median()))

In [134]:
# 3. Handle Missing Quantities and Cast to Integer
# Fill missing quantities with 1 (baseline assumption for a transaction) and convert to int
df['quantity'] = df['quantity'].fillna(1).astype(int)

# Check the results of our cleaning phase
print("--- Updated Dataset Info After Phase 1 Cleaning ---")
print(df.info())

print("\n--- Remaining Missing Values ---")
print(df.isna().sum())

print("\n--- Sample of Cleaned Price and Quantity Data ---")
print(df[['product_name', 'price', 'price_cleaned', 'quantity']].head(10))

--- Updated Dataset Info After Phase 1 Cleaning ---
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_id        15000 non-null  int64  
 1   date           15000 non-null  str    
 2   product_name   15000 non-null  str    
 3   price          14579 non-null  str    
 4   quantity       15000 non-null  int64  
 5   price_cleaned  15000 non-null  float64
dtypes: float64(1), int64(2), str(3)
memory usage: 703.3 KB
None

--- Remaining Missing Values ---
sale_id            0
date               0
product_name       0
price            421
quantity           0
price_cleaned      0
dtype: int64

--- Sample of Cleaned Price and Quantity Data ---
      product_name     price  price_cleaned  quantity
0      MacBook Pro      1999        1999.00         1
1      macbook air      1100        1100.00         3
2         iPad Air   599 USD         599.00     

## Step 3: Standardizing Date Formats
The original dataset contains dates in four different formats due to system inconsistencies. In this step, we use Pandas `to_datetime` with `errors='coerce'` and alternative format parsing to uniform all values into a standard `YYYY-MM-DD` datetime format, allowing for proper chronological analysis.

In [135]:
# 4. Standardize Date Column
# We try to parse dates automatically. Pandas 2.0+ is smart, but with mixed formats, 
# converting to string and using pd.to_datetime with format='mixed' handles the chaos perfectly.
df['date_cleaned'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')

# Check if we have any unparsed dates left
unparsed_dates = df['date_cleaned'].isna().sum()
print(f"Number of dates that failed to parse: {unparsed_dates}")

# Extract year and month for future business grouping
df['year_month'] = df['date_cleaned'].dt.to_period('M')

print("\n--- Sample of Original vs Cleaned Dates ---")
print(df[['date', 'date_cleaned', 'year_month']].head(10))

Number of dates that failed to parse: 0

--- Sample of Original vs Cleaned Dates ---
               date        date_cleaned year_month
0  2026/01/04 18:00 2026-01-04 18:00:00    2026-01
1  2026/01/20 15:00 2026-01-20 15:00:00    2026-01
2        07/01/2026 2026-07-01 00:00:00    2026-07
3  2026/01/11 07:00 2026-01-11 07:00:00    2026-01
4        2026-01-09 2026-01-09 00:00:00    2026-01
5        11/01/2026 2026-11-01 00:00:00    2026-11
6        05/01/2026 2026-05-01 00:00:00    2026-05
7  2026/01/16 00:00 2026-01-16 00:00:00    2026-01
8        2026-01-09 2026-01-09 00:00:00    2026-01
9        2026-01-21 2026-01-21 00:00:00    2026-01


## Step 4: Final Text Standardization
Before calculating financial metrics, we ensure all product names are perfectly uniform. We capitalize them correctly so that variations like 'iphone 15 pro' and 'iPhone 15 Pro' are grouped as the exact same product.

In [136]:
# Standardize the capitalization of products to avoid duplicate groupings
# We can map everything to a clean, standard look
product_mapping = {
    'iphone 15 pro': 'iPhone 15 Pro',
    'macbook air': 'MacBook Air',
    'ipad air': 'iPad Air',
    'MacBook Air': 'MacBook Air',
    'iPad Air': 'iPad Air',
    'MacBook Pro': 'MacBook Pro',
    'AirPods Pro': 'AirPods Pro',
    'Powerbank 65W': 'Powerbank 65W',
    'Wi-Fi Adapter': 'Wi-Fi Adapter',
    'Unknown Product': 'Unknown Product'
}

# Apply mapping after stripping any remaining accidental edge cases
df['product_name_cleaned'] = df['product_name'].str.strip().map(lambda x: product_mapping.get(x, x))

# Verify unique products left
print("--- Cleaned Unique Products List ---")
print(df['product_name_cleaned'].unique())

--- Cleaned Unique Products List ---
<StringArray>
[    'MacBook Pro',     'MacBook Air',        'iPad Air', 'Unknown Product',
   'Wi-Fi Adapter',   'iPhone 15 Pro',   'Powerbank 65W',     'AirPods Pro']
Length: 8, dtype: str


## Step 5: Financial Metrics and Business Intelligence Reporting
With clean and standardized data, we now calculate the `total_revenue` for each transaction ($Price \times Quantity$). We then aggregate the dataset to generate two critical business insights: monthly revenue trends and overall product performance.

In [137]:
# 1. Calculate Total Revenue for each row
df['total_revenue'] = df['price_cleaned'] * df['quantity']

In [138]:
# 2. Aggregate Data: Monthly Sales Performance
monthly_report = df.groupby('year_month').agg(
    total_sales_volume=('quantity', 'sum'),
    total_revenue_usd=('total_revenue', 'sum'),
    average_ticket=('total_revenue', 'mean')
).reset_index()

In [139]:
# 3. Aggregate Data: Product Sales Performance (Sorted by highest revenue)
product_report = df.groupby('product_name_cleaned').agg(
    units_sold=('quantity', 'sum'),
    revenue_generated_usd=('total_revenue', 'sum')
).sort_values(by='revenue_generated_usd', ascending=False).reset_index()

# Format financial figures for clean professional presentation in outputs
monthly_report['total_revenue_usd'] = monthly_report['total_revenue_usd'].map('${:,.2f}'.format)
monthly_report['average_ticket'] = monthly_report['average_ticket'].map('${:,.2f}'.format)
product_report['revenue_generated_usd'] = product_report['revenue_generated_usd'].map('${:,.2f}'.format)

print("=== MONTHLY BUSINESS PERFORMANCE REPORT ===")
print(monthly_report.to_string(index=False))

print("\n=== TOP PRODUCTS BY REVENUE REPORT ===")
print(product_report.to_string(index=False))

=== MONTHLY BUSINESS PERFORMANCE REPORT ===
year_month  total_sales_volume total_revenue_usd average_ticket
   2026-01               14249     $9,917,561.57      $2,073.94
   2026-02               13180     $9,554,910.13      $2,155.89
   2026-03               13756     $9,978,155.02      $2,150.00
   2026-04                 345       $239,507.68      $2,047.07
   2026-05                 421       $309,068.50      $2,207.63
   2026-06                 438       $294,064.58      $2,000.44
   2026-07                 371       $229,541.45      $1,881.49
   2026-08                 351       $248,444.58      $2,087.77
   2026-09                 354       $261,379.48      $2,354.77
   2026-10                 420       $343,777.39      $2,565.50
   2026-11                 419       $290,656.43      $2,153.01
   2026-12                 357       $274,667.71      $2,288.90

=== TOP PRODUCTS BY REVENUE REPORT ===
product_name_cleaned  units_sold revenue_generated_usd
       iPhone 15 Pro        8

## Step 6: Exporting Processed Data to Multi-Sheet Excel File
Finally, we save our clean master dataset along with the generated business intelligence reports into a single, structured Excel workbook. This allows stakeholders to easily navigate between raw transactional data and summarized executive insights.

In [140]:
# 1. Створюємо функцію для гнучкого парсингу дат (Flexible Date Parsing)
def clean_mixed_dates(date_series):
    # Спочатку переводимо все в рядок і очищаємо від залишків часу (наприклад, 18:00)
    parsed_dates = pd.to_string = date_series.astype(str).str.split(' ').str[0]
    
    # Створюємо порожню серію для результатів
    final_dates = pd.to_datetime(pd.Series([None] * len(date_series)), errors='coerce')
    
    # Список форматів, які замовник намішав у файлах, у порядку пріоритету
    formats_to_try = [
        '%Y-%m-%d',  # 2026-01-15
        '%Y/%m/%d',  # 2026/01/15
        '%d/%m/%Y',  # 15/01/2026
        '%m-%d-%Y'   # 01-15-2026
    ]
    
    # Проходимо по кожному формату: заповнюємо лише те, що ще не розпарсилося
    for fmt in formats_to_try:
        still_nan = final_dates.isna()
        if not still_nan.any():
            break
        # Пробуємо поточний формат для проблемних рядків
        attempt = pd.to_datetime(parsed_dates[still_nan], format=fmt, errors='coerce')
        final_dates.update(attempt)
        
    return final_dates

# Запускаємо наш розумний парсер
df['date_cleaned'] = clean_mixed_dates(df['date'])

# Перевіряємо, чи є тепер пропуски. Має бути 0!
unparsed_count = df['date_cleaned'].isna().sum()
print(f"Кількість дат, які НЕ вдалося розпарсити: {unparsed_count}")

# Перераховуємо місяці та загальний дохід
df['year_month'] = df['date_cleaned'].dt.to_period('M')
df['total_revenue'] = df['price_cleaned'] * df['quantity']

# 2. Перегенерація бізнес-звітів
monthly_summary = df.groupby('year_month').agg(
    total_sales_volume=('quantity', 'sum'),
    total_revenue_usd=('total_revenue', 'sum'),
    average_ticket=('total_revenue', 'mean')
).sort_values(by='year_month').reset_index()
monthly_summary['year_month'] = monthly_summary['year_month'].astype(str)

product_summary = df.groupby('product_name_cleaned').agg(
    units_sold=('quantity', 'sum'),
    revenue_generated_usd=('total_revenue', 'sum')
).sort_values(by='revenue_generated_usd', ascending=False).reset_index()

# Готуємо Master Data для експорту
master_clean_export = df[[
    'sale_id', 'date_cleaned', 'product_name_cleaned', 'price_cleaned', 'quantity', 'total_revenue'
]].copy()
master_clean_export.columns = ['Sale ID', 'Date', 'Product Name', 'Unit Price', 'Quantity', 'Total Revenue']
master_clean_export['Date'] = master_clean_export['Date'].dt.strftime('%Y-%m-%d')

# 3. Запис у Excel зі стилізацією
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    from openpyxl.styles import Font, PatternFill

output_filename = "final_sales_analytics_report.xlsx"

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    master_clean_export.to_excel(writer, sheet_name="Cleaned Master Data", index=False)
    monthly_summary.to_excel(writer, sheet_name="Monthly Performance", index=False)
    product_summary.to_excel(writer, sheet_name="Product Performance", index=False)
    
    workbook = writer.book
    header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    regular_font = Font(name='Segoe UI', size=11)
    currency_format = '$#,##0.00'
    integer_format = '#,##0'

    # Стиль для Cleaned Master Data
    ws_master = workbook["Cleaned Master Data"]
    ws_master.freeze_panes = "A2"
    for cell in ws_master[1]:
        cell.font = header_font
        cell.fill = header_fill
    for row in range(2, ws_master.max_row + 1):
        for col in range(1, 4):
            ws_master.cell(row=row, column=col).font = regular_font
        ws_master.cell(row=row, column=4).number_format = currency_format
        ws_master.cell(row=row, column=4).font = regular_font
        ws_master.cell(row=row, column=5).number_format = integer_format
        ws_master.cell(row=row, column=5).font = regular_font
        ws_master.cell(row=row, column=6).number_format = currency_format
        ws_master.cell(row=row, column=6).font = regular_font

    # Стиль для Monthly Performance
    ws1 = workbook["Monthly Performance"]
    for cell in ws1[1]:
        cell.font = header_font
        cell.fill = header_fill
    for row in range(2, ws1.max_row + 1):
        ws1.cell(row=row, column=1).font = regular_font
        ws1.cell(row=row, column=2).number_format = integer_format
        ws1.cell(row=row, column=2).font = regular_font
        ws1.cell(row=row, column=3).number_format = currency_format
        ws1.cell(row=row, column=3).font = regular_font
        ws1.cell(row=row, column=4).number_format = currency_format
        ws1.cell(row=row, column=4).font = regular_font

    # Стиль для Product Performance
    ws2 = workbook["Product Performance"]
    for cell in ws2[1]:
        cell.font = header_font
        cell.fill = header_fill
    for row in range(2, ws2.max_row + 1):
        ws2.cell(row=row, column=1).font = regular_font
        ws2.cell(row=row, column=2).number_format = integer_format
        ws2.cell(row=row, column=2).font = regular_font
        ws2.cell(row=row, column=3).number_format = currency_format
        ws2.cell(row=row, column=3).font = regular_font

    # Авто-ширина
    for ws in [ws_master, ws1, ws2]:
        for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = openpyxl.utils.get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

print(f"[SUCCESS] Експорт завершено! Файл '{output_filename}' оновлено.")


Кількість дат, які НЕ вдалося розпарсити: 0
[SUCCESS] Експорт завершено! Файл 'final_sales_analytics_report.xlsx' оновлено.
